# PyMIF beginner notebook 04 - read Zarr and inspect raw, groups, and labels

This notebook creates a small example Zarr with raw data, one image group, and one label group, then reads it back.

This is the most important notebook for understanding the new Zarr layout:

- `z.raw` is the root image.
- `z.groups` contains root-level image groups such as `denoised` or `probability`.
- `z.labels` contains label groups under `/labels` such as `nuclei`.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

In [ ]:
def make_small_raw_manager(seed=0, num_levels=2):
    """Create a tiny TCZYX intensity dataset for examples."""
    rng = np.random.default_rng(seed)
    raw = rng.integers(0, 1200, size=(2, 3, 4, 64, 64), dtype=np.uint16)
    metadata = {
        "name": "small_raw",
        "axes": "tczyx",
        "size": [raw.shape],
        "chunksize": [(1, 1, 2, 32, 32)],
        "scales": [(2.0, 0.30, 0.30)],   # z, y, x spacing because axes contains zyx
        "units": ("micrometer", "micrometer", "micrometer"),
        "time_increment": 60.0,
        "time_increment_unit": "second",
        "channel_names": ["DAPI", "GFP", "RFP"],
        "channel_colors": ["0000FF", "00FF00", "FF0000"],
        "dtype": "uint16",
        "data_type": "intensity",
    }
    manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
    if num_levels > 1:
        manager.build_pyramid(num_levels=num_levels, downscale_factor=(1, 2, 2))
    return manager

In [ ]:
def label_metadata_from_image_metadata(image_metadata, name="nuclei", dtype="uint32"):
    """Make safe label metadata from image metadata by removing the channel axis.

    Most segmentation labels are one integer label image per time point and do
    not have an image channel axis. This helper avoids confusion from legacy
    TCZYX examples by creating explicit TZYX label metadata.
    """
    md = dict(image_metadata)
    axes = str(md["axes"]).lower()
    if "c" in axes:
        c_axis = axes.index("c")
        md["axes"] = "".join(ax for ax in axes if ax != "c")
        md["size"] = [tuple(v for i, v in enumerate(size) if i != c_axis) for size in md["size"]]
        md["chunksize"] = [tuple(v for i, v in enumerate(chunks) if i != c_axis) for chunks in md["chunksize"]]
    md["name"] = name
    md["dtype"] = dtype
    md["data_type"] = "label"
    md["is_label"] = True
    md["channel_names"] = []
    md["channel_colors"] = []
    return md

## Create a small example store

In [ ]:
raw_manager = make_small_raw_manager(seed=7, num_levels=2)
example_path = clean_path(OUTPUT_DIR / "read_inspect_example.zarr")
raw_manager.to_zarr(example_path, ngff_version="0.5", zarr_format=3)

z = mm.ZarrManager(example_path, mode="r+")

# Add an image group with the same metadata shape as raw.
denoised = z.create_empty_group("denoised", dict(z.metadata), is_label=False)
denoised_data = np.clip(raw_manager.data[0].compute() // 2, 0, 65535).astype(np.uint16)
z.write_image_region(denoised_data, group="denoised", level=0, downscale_factor=(1, 2, 2))

# Add a label group under /labels/nuclei.
label_md = label_metadata_from_image_metadata(z.metadata, name="nuclei", dtype="uint32")
z.create_empty_group("nuclei", label_md, is_label=True)
label_data = np.zeros(label_md["size"][0], dtype=np.uint32)
label_data[:, :, 15:40, 20:45] = 1
label_data[:, :, 45:58, 48:60] = 2
z.write_label_region(label_data, group="labels/nuclei", level=0, downscale_factor=(1, 2, 2))

z.read()
print("Example store ready:", example_path)

## Inspect the hierarchy

`ZarrManager.read()` prints the Zarr tree. You can also inspect the structured objects directly.

In [ ]:
z = mm.ZarrManager(example_path, mode="r")
print("\nRoot/raw object:", z.raw)
print("Image group names:", list(z.groups.keys()))
print("Label group names:", list(z.labels.keys()))

summarize_manager(z, "root image via old aliases")
print("\nDenoised dataset:", z.groups["denoised"])
print("Denoised level 0 shape:", z.groups["denoised"].data[0].shape)
print("\nNuclei labels:", z.labels["nuclei"])
print("Nuclei level 0 shape:", z.labels["nuclei"].data[0].shape)

## Read a small piece without loading the whole dataset

Dask keeps data lazy. Slice first, then call `.compute()` on a small region.

In [ ]:
raw_small = z.raw.data[0][0, 0, 0, :8, :8].compute()
label_small = z.labels["nuclei"].data[0][0, 0, :8, :8].compute()

print("raw small region shape:", raw_small.shape, "dtype:", raw_small.dtype)
print(raw_small)
print("\nlabel small region shape:", label_small.shape, "dtype:", label_small.dtype)
print(label_small)

## Safe pattern for downstream code

Prefer writing code against the explicit objects. Use the old aliases only when you mean the root image.

In [ ]:
# Recommended:
raw_level0 = z.raw.data[0]
all_image_groups = z.groups
all_label_groups = z.labels

# Legacy-compatible but less explicit:
legacy_root_level0 = z.data[0]

print(raw_level0.shape, legacy_root_level0.shape)